# Select sample size for PM<sub>2.5</sub> TMREL, RR and BMR

Selecting the number of samples is a balance between computational cost, given the high spatial resolution and associated memory requirements, and the need for a sufficiently large sample size to robustly span the uncertainty space.

The example we use here is for 300 samples.

In [ ]:
import os
import glob
import xarray as xr
import numpy as np
# create_global_country_map includes a file path to country mask
from utils.utils import create_global_country_map

In [ ]:
# === CHOOSE NUMBER OF SAMPLES ===
n_samples = 300

# Set the seed to produce reproducible random numbers
np.random.seed(42)

In [ ]:
# === Calculate the TMREL distribution ===

# TMREL from GBD21 (uniform distribution)
tmrel_low = 2.4
tmrel_high = 5.9
tmrel_samples = np.random.uniform(tmrel_low, tmrel_high, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=["samples"],
    coords={"samples": np.arange(n_samples)}
).astype("float32")

# === Save file in scratch directory ===
SAVE_DIR = "/glade/derecho/scratch/awells/air_quality/TMREL/"

out_file = f"TMREL_{n_samples}_samples_pm25.nc"
out_path = os.path.join(SAVE_DIR, out_file)
tmrel_da.to_netcdf(out_path)

In [ ]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [ ]:
# === Calculate the RR distribution ===
# Scaled to the TMREL so that RR=1 below the TMREL

RR_DIR = "/glade/work/awells/workflow/GBD23/RR_curves/"

for health_VAR in health_vars:
    print(f"Processing {health_VAR}")
    pattern = os.path.join(RR_DIR, f"IHME_GBD_2023_AIR_POLLUTION_*_PM_RR_{health_VAR}.nc")
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(f"No .nc file found for variable: {health_VAR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple .nc files matched for variable {health_VAR}: {matches}")
    RR = xr.open_dataset(matches[0])

    # Updated GBD23 risk curves are log(RR), non updated curves are RR
    if RR["mean"][0] == 1:
        print("Data starts at 1 so they are 'Relative Risk'")
        # Calculate log(RR)
        logRR = np.log(RR)
    elif RR["mean"][0] == 0:
        print("Data starts at 0 so they are 'log(Relative Risk)'")
        # Data is already in logRR format
        logRR = RR
    else:
        raise ValueError(f"Data has unknown start value: {RR['mean'][0]}")

    # Calculate the normal distribution for the logRR
    logRR_mean = logRR["mean"]
    logRR_lower = logRR["lower"]
    logRR_upper = logRR["upper"]

    logRR_std = (logRR_upper - logRR_lower) / (2 * 1.96)

    logRR_samples = xr.DataArray(
        np.random.normal(
            logRR_mean.values[..., np.newaxis],
            logRR_std.values[..., np.newaxis],
            size=logRR_mean.shape + (n_samples,)
        ),
        dims=logRR_mean.dims + ("samples",),
        coords={**logRR_mean.coords, "samples": np.arange(n_samples)},
    )

    # Find the log(RR) at the TMREL
    logRR_tmrel = logRR_samples.sel(exposure=tmrel_da, method="nearest")

    # Shift the function by the log(RR)_TMREL so that log(RR)=0 at TMREL
    logRR_shifted = logRR_samples - logRR_tmrel

    # Set log(RR) below TMREL as 0 and exponentiate to get RR
    RR_samples = np.exp(logRR_shifted.where(logRR_shifted["exposure"] >= tmrel_da, 0))

    # === Save file in scratch directory ===
    SAVE_DIR = "/glade/derecho/scratch/awells/workflow/rr_pm25/"

    out_file = f"GBD23_RR_{health_VAR}_{n_samples}_samples_pm25.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    RR_samples.to_netcdf(out_path)

print("All processing complete.")

In [ ]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

# Save file in scratch directory ~25GB
SAVE_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"

In [ ]:
# === Calculate the BMR distribution ===

for health_VAR in health_vars:
    print(f"Processing {health_VAR}")

    # Load BMR for each country (lat, lon, quantile)
    bmr_file = f"GBD23_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # three quantiles

    # BMR from VizHub (normal distribution)
    bmr_mean = BMR.sel(quantile="mean")
    bmr_lower = BMR.sel(quantile="lower")
    bmr_upper = BMR.sel(quantile="upper")
    bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

    bmr_samples = np.random.normal(
        bmr_mean,
        bmr_std,
        size=(n_samples, len(BMR.country)))

    bmr_da = xr.DataArray(
        bmr_samples,
        dims=["samples", "country"],
        coords={"samples": np.arange(n_samples), "country": BMR.country}
    ).astype("float32")

    del bmr_samples

    # WARNING: this step can be slow and use a lot of memory
    # e.g. ~100GB for 1000 samples, ~30GB for 200 samples
    bmr_global = create_global_country_map(bmr_da).chunk({"samples": 10, "lat": 180, "lon": 360})

    # Save file ~25GB
    out_file = f"GBD23_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_1990-2009.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    bmr_global.to_netcdf(out_path)

    del bmr_global

print("All processing complete.")